# AIR Homework 1 (Task 2)

## Setup

This notebook covers Task 2. Run the cells below before you start. They set up the notebook, load our working environment from `scene.py`, import your code, and recreate what we need from Task 1.

**NOTE:** The setup uses your code in `task1.py`, so complete Task 1 before you start this task.

In [ ]:
%load_ext autoreload
%autoreload 2

# set plot theme
import seaborn as sns
sns.set_theme()

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

In [ ]:
# load our working environment from scene.py (see start_here.ipynb)
from scene import data, model

import mujoco

In [ ]:
# import your code. the setup below uses your code from Task 1 (t1), and the rest
# of this notebook uses your code for this task (t2), e.g. t2.get_entity_ctrl_idx
import task1 as t1
import task2 as t2

In [ ]:
# from Task 1
import unittest

# used by the sanity checks
test = unittest.TestCase()

# list camera options
CAMERA_LIST = [
    "leftsideview",
    "rightsideview",
    "frontview",
    "birdview",
    "leftangleview",
    "rightangleview",
]

# the pre-grasp pose above the red box, at the home keyframe
mujoco.mj_resetDataKeyframe(model, data, model.key("robot_home").id)
mujoco.mj_forward(model, data)
above_block_pos, above_block_rot_mat = t1.get_above_block_pose(data)

# compute IK solutions for the above-box end-effector pose
box_ik_solutions = t1.compute_ik(data, above_block_pos, above_block_rot_mat)

# use the same collision-free solution you selected at the end of Task 1
above_box_config = box_ik_solutions[0]

## Task 2: PID Controller in Configuration Space

## Proportional Control for Motor Actuation

Motor actuators apply torque to the joints of the robot. Unlike position actuators, which directly set the joint angles, motor actuators require us to compute the necessary torques to achieve the desired joint angles.

What is torque? Torque is a measure of the rotational force applied to an object. In the context of robotic joints, torque determines how much force is applied to rotate a joint around its axis. Consequently, it is proportional to the angular acceleration. That means applying force in the direction of the desired movement will cause the joint to accelerate towards the target position.

We are now ready to implement our controller. First, let's implement the abstract policy class and rendering function as before:

`Policy` and `run_and_render_policy` are provided in `task2.py`.

Now, let's bring over our implementation of the proportional controller from the previous tutorial and modify it to work in configuration space with motor actuation.

As we did earlier with `qpos`, let's write a helper function to find the indices of the robot's `ctrl` values. These will be the IDs of the actuators that correspond to the robot (obtained through actuator named access).

**NOTE:** The number of actuators available can be found in `model.nu`.

**TODO:** implement `get_entity_ctrl_idx` in `task2.py`.

In [ ]:
# sanity check: get_entity_ctrl_idx
robot_ctrl_ids = list(map(int, t2.get_entity_ctrl_idx(model, "robot")))
gripper_ctrl_ids = list(map(int, t2.get_entity_ctrl_idx(model, "gripper")))

# one actuator for each robot joint, and a single actuator for the whole gripper
test.assertEqual(robot_ctrl_ids, [0, 1, 2, 3, 4, 5])
test.assertEqual(gripper_ctrl_ids, [6])

print(f"get_entity_ctrl_idx sanity checks passed. robot = {robot_ctrl_ids}, gripper = {gripper_ctrl_ids}")

And now the controller itself.

**Hint**: It's actually more natural and intuitive to implement the controller in terms of joint motors rather than end-effector position. Don't overthink it!

**TODO:** implement `ProportionalJointPositionControlPolicy` in `task2.py`.

Let's bring the robot to the home configuration and visualize the different solutions using different proportional gain values.

**NOTE**: We fully expect the robot to knock over the obstacles. That's okay! We will address this in the next task using motion planning.

In [ ]:
from ipywidgets import interact
import numpy as np

# the robot's home configuration
from scene import HOME_CONFIG


# interactive visualization of proportional control policy
@interact(kp=list(np.arange(0, 1, 0.01)), camera=CAMERA_LIST)
def render_proportional_control(kp=0.1, camera="frontview"):
    home_config_policy = t2.ProportionalJointPositionControlPolicy(
        model, target_qpos=HOME_CONFIG, kp=kp
    )
    t2.run_and_render_policy(
        model, data, home_config_policy, duration=2.0, framerate=60, camera=camera
    )

We can see that it is very difficult to stabilize the control motion when using motor control. Very small changes in the proportional gain can lead to drastic differences in the behavior of the system. Let's visualize this another way by plotting the distance to the target configuration over time for different gain values:

`TargetDiffCallback` is provided in `task2.py`.

In [ ]:
import matplotlib.pyplot as plt

# a container to store the target differences for different kp values
p_diffs_data = {}

# run simulations for different kp values and collect target differences
for i in [0.01, 0.02, 0.03, 0.04, 0.05, 1, 50]:
    # a callback to record target differences
    cb = t2.TargetDiffCallback()

    # create a proportional control policy with the current kp value
    home_config_policy = t2.ProportionalJointPositionControlPolicy(
        model, target_qpos=HOME_CONFIG, kp=i
    )

    # run the simulation with the policy and callback.
    # set framerate to 0 to skip rendering during data collection
    t2.run_and_render_policy(
        model, data, home_config_policy, duration=5.0, framerate=0, callback=cb
    )

    # store the collected target differences from the callback for this kp value
    p_diffs_data[i] = cb.target_diffs

# plot the target differences over time for different kp values
plt.figure()
for i in p_diffs_data:
    plt.plot(p_diffs_data[i], label=f"kp={i}")
plt.hlines(0, 0, len(p_diffs_data[i]), colors="k", linestyles="dashed")
plt.xlabel("Timestep")
plt.ylabel("L2 Distance to Target Position")
plt.title("L2 Distance to Target Position over Time for Different Kp Values")
plt.legend()
plt.show()

As we can see, proportional control is insufficient. It is far too difficult to tune the gain to achieve stable control. The system either stabilizes far from the target configuration or overshoots the target and oscillates uncontrollably. We need something else to help us stabilize the system.

## PID Control

PID stands for Proportional-Integral-Derivative control. It is an extension of the proportional controller that we have already implemented. The idea is to add two more terms to the control signal to improve performance:
1. **Integral Term**: The integral of the error accounts for the accumulation of past errors. It helps eliminate steady-state errors that may occur with a pure proportional controller.
2. **Derivative Term**: The derivative of the error predicts future errors based on the current rate of change of the error. It helps dampen the system's response and reduce overshooting.

The control signal for a PID controller at time $t$ is given by:
$$
u(t) = kp \cdot e(t) + ki \cdot \int_0^t e(\tau) d\tau + kd \cdot \frac{\partial}{\partial t} e(t)
$$

We already know how to calculate the error term:
$$
e(t) = r - q(t)
$$
Where $r$ is the reference (target configuration) and $q(t)$ is the robot configuration at time $t$.

The derivative of the error is calculated as follows:
$$
\frac{\partial}{\partial t} e(t)= \frac{\partial}{\partial t} (r - q(t)) = -\frac{\partial}{\partial t} q(t) = -v(t)
$$
Where $v(t)$ is the current joint velocities at time $t$.

The integral of the error is calculated in discrete time as follows:
$$
\int_0^t e(\tau) d\tau \approx \sum_{i=0}^{n} e(t_i) \Delta t
$$

where $t_n = t$ is the current timestep. However, in practice, the integral term can lead to a phenomenon known as "integral windup." This occurs when the integral term accumulates a significant error during periods when the controller is saturated (i.e., when the control signal exceeds the actuator limits). To mitigate this, we can implement a simple anti-windup mechanism by adding a decay term $\alpha \in [0, 1]$ to the integral calculation:
$$
\int_0^t e(\tau) d\tau \approx \sum_{i=0}^{n} \alpha^{n - i} e(t_i) \Delta t
$$

$$

Let's start by getting the velocity indices for the robot, as this is needed for the derivative term. These are equivalent to the degrees of freedom for the UR5e robot, found in `model.joint(JOINT_ID).dofadr`:

**TODO:** implement `get_entity_qvel_idx` in `task2.py`.

In [ ]:
# sanity check: get_entity_qvel_idx
robot_qvel_ids = list(map(int, t2.get_entity_qvel_idx(model, "robot")))

# these are not the same as the qpos indices. the five free joints that come before the
# robot have seven qpos entries each, but only six qvel entries
test.assertEqual(robot_qvel_ids, [30, 31, 32, 33, 34, 35])

print(f"get_entity_qvel_idx sanity checks passed. robot = {robot_qvel_ids}")

Now, implement the PID controller:

**TODO:** implement `PIDJointPositionControlPolicy` in `task2.py`.

## Understanding PID Gains

The proportional gain is fairly straightforward: it is the proportion of the error that we use as the desired velocity. Now, let's understand the role of the derivative term and its gain.

In [ ]:
# a container to store the target differences for different kd values
pd_diffs_data = {}

# run simulations for different kd values and collect target differences
for i in [0, 0.1, 0.5, 1, 5, 10, 50, 100]:
    # a callback to record target differences
    cb = t2.TargetDiffCallback()

    # create a PID control policy with the current kd value
    home_config_policy = t2.PIDJointPositionControlPolicy(
        model, target_qpos=HOME_CONFIG,
        kp=1, ki=0, kd=i
    )

    # set framerate to 0 to skip rendering during data collection
    t2.run_and_render_policy(model, data, home_config_policy, duration=5.0, framerate=0, callback=cb)
    pd_diffs_data[i] = cb.target_diffs

# plot the target differences over time for different kd values
plt.figure()
for i in pd_diffs_data:
    plt.plot(pd_diffs_data[i], label=f'kd={i}')
plt.hlines(0, 0, len(pd_diffs_data[i]), colors='k', linestyles='dashed')
plt.xlabel('Timestep')
plt.ylabel('Target Position Difference Norm')
plt.title('Target Position Difference over Time for Different Kd Values')
plt.legend()
plt.show()

The derivative term is a form of damping. It adds a velocity that is opposite to the current velocity, scaled by the derivative gain. This means that if the robot is moving quickly towards the target, the derivative term will apply a strong opposing velocity to slow it down. Conversely, if the robot is moving slowly, the opposing velocity will be smaller. Higher gains mean stronger damping effects.

With gain $kd=1$, The robot converges to the target, but not very smoothly. This is because the damping effect is too weak to counteract the momentum of the robot as it approaches the target configuration. With $kd=100$, the robot converges very smoothly but far from the target. This is because the damping effect is too strong, preventing the robot from reaching the target configuration effectively. With $kd=10$, we achieve a good balance: the robot converges to the target smoothly without overshooting. However, we can see that the robot is not quite reaching the target configuration.

Now let's understand the integral term and its gain.

In [ ]:
# a container to store the target differences for different ki values
pid_diffs_data = {}

# run simulations for different ki values and collect target differences
for i in [0, 0.1, 0.5, 1, 5, 10, 50, 100]:
    # a callback to record target differences
    cb = t2.TargetDiffCallback()

    # create a PID control policy with the current ki value
    home_config_policy = t2.PIDJointPositionControlPolicy(
        model, target_qpos=HOME_CONFIG,
        kp=1, ki=i, kd=0.1
    )

    # set framerate to 0 to skip rendering during data collection
    t2.run_and_render_policy(model, data, home_config_policy, duration=5.0, framerate=0, callback=cb)
    pid_diffs_data[i] = cb.target_diffs

# plot the target differences over time for different ki values
plt.figure()
for i in pid_diffs_data:
    plt.plot(pid_diffs_data[i], label=f'ki={i}')
plt.hlines(0, 0, len(pid_diffs_data[i]), colors='k', linestyles='dashed')
plt.xlabel('Timestep')
plt.ylabel('Target Position Difference Norm')
plt.title('Target Position Difference over Time for Different Ki Values')
plt.legend()
plt.show()

The integral term helps eliminate steady-state errors. If the robot is consistently falling short of the target configuration, the integral term will accumulate this error over time and apply a torque that pushes the robot closer to the target. Higher gains mean that the integral term will have a stronger effect on the control signal.

Let's compare the three controllers we have implemented so far (proportional, PD, and PID) by finding the appropriate gain for each.

### Gain Tuning

Now that we understand the role of each term in the PID controller, we would like to choose the gains that work best for our system. While there exist sophisticated methods for tuning PID gains, we will use a simple grid-search approach.

The code below runs a series of simulations with different PID gain values and collects the distance to the target configuration over time. Play around with the `kp`, `ki`, and `kd` value ranges in the cell below to find a set of gains that work well for going to the home configuration.

**This may take a few minutes to run**

In [ ]:
from itertools import product
from tqdm.auto import tqdm

# select tuning values
KP_RANGE = [0.1, 0.5, 1, 5, 10, 50, 100]
KI_RANGE = [0, 0.1, 0.5, 1, 5, 10, 50, 100]
KD_RANGE = [0, 0.1, 0.5, 1, 5, 10, 50, 100]

# track progress
pbar = tqdm(total=len(KP_RANGE) * len(KI_RANGE) * len(KD_RANGE), desc="Gain Tuning")

# collect data for all combinations of kp, ki, kd
full_pid_diffs_data = {}

for kp, ki, kd in product(KP_RANGE, KI_RANGE, KD_RANGE):
    cb = t2.TargetDiffCallback()
    home_config_policy = t2.PIDJointPositionControlPolicy(
        model, target_qpos=HOME_CONFIG,
        kp=kp, ki=ki, kd=kd, integral_decay=0.99
    )

    # set framerate to 0 to skip rendering during data collection
    t2.run_and_render_policy(model, data, home_config_policy, duration=20.0, framerate=0, callback=cb)
    full_pid_diffs_data[(kp, ki, kd)] = cb.target_diffs
    pbar.update(1)

We choose the best parameters based on the results of the grid search. The selected PID gains will achieve the minimum distance to the target configuration in the shortest amount of time without extreme oscillations or overshooting. Though not entirely reliable, we detect oscillations by checking that the target position distance is "approximately" monotonically decreasing. We define what "approximately" means with the threshold value `EPSILON`, which **you need to tune yourself** in `task2.py`: a value that is too small rejects every controller as non-monotone, while a value that is too large accepts controllers that oscillate. After changing it, rerun the cell below. If the controller has no results that satisfy this condition, we fall back to the best performing parameters based on minimum distance alone (but indicate that oscillations were detected).

We find the best gain values for P-only, PD, and PID controllers and present them below:

In [ ]:
# tune EPSILON in task2.py

# filter data for PD and P only controllers
full_pd_diffs_data = {
    (kp, ki, kd): v for (kp, ki, kd), v in full_pid_diffs_data.items() if ki == 0
}  # filter to PD only
full_p_diffs_data = {
    (kp, ki, kd): v for (kp, ki, kd), v in full_pd_diffs_data.items() if kd == 0
}  # filter to P only

# get and present best parameters for P only, PD, and PID controllers
params_p, min_diff_p, min_time_p, found_monotone_params_p = t2.get_best_params(
    full_p_diffs_data
)
params_pd, min_diff_pd, min_time_pd, found_monotone_params_pd = t2.get_best_params(
    full_pd_diffs_data
)
best_params, min_diff, min_time, found_monotone_params = t2.get_best_params(
    full_pid_diffs_data
)

# present the results
t2.present_best_params(
    params_p, min_diff_p, min_time_p, found_monotone_params_p, label="P only"
)
t2.present_best_params(
    params_pd, min_diff_pd, min_time_pd, found_monotone_params_pd, label="PD"
)
t2.present_best_params(best_params, min_diff, min_time, found_monotone_params, label="PID")

Below we visualize the performance of each controller with the selected gains. You may need to zoom in to see the differences clearly.

In [ ]:
# plot comparison of best parameters
# allow interactive zooming toward the lower error region
@interact(zoom=[None, 1, 0.5, 0.25, 0.1, 0.05])
def plot_compare_pid_params(zoom=None):
    plt.figure()
    plt.plot(full_pid_diffs_data[params_p], label="P only")
    plt.plot(full_pid_diffs_data[params_pd], label="PD")
    plt.plot(full_pid_diffs_data[best_params], label="PID")
    plt.hlines(0, 0, len(next(iter(full_pd_diffs_data.values()))), colors='k', linestyles='dashed')
    plt.xlabel('Timestep')
    plt.ylabel('Target Position Difference Norm')
    plt.title('Target Position Difference over Time for Best Gain Parameters')
    plt.legend()

    if zoom is not None:
        plt.ylim(0, zoom)

    plt.show()

Let's test out our new PID controller by bringing the robot to the grasping configuration above the red box.

Set the gains to the selected PID values in `task2.py` and run the cell below. These will be the gains we use for the rest of the assignment. Keep in mind that in practice, it often makes sense to have different parameters for different parts of the task. For example, we may prefer a more aggressive controller when in a clear space, and a more conservative controller when around obstacles.

In [ ]:
# set KP, KI, and KD in task2.py

# create a PID control policy to move above the box and run the simulation
above_box_policy = t2.PIDJointPositionControlPolicy(
    model, target_qpos=above_box_config,
    kp=t2.KP, ki=t2.KI, kd=t2.KD
)
t2.run_and_render_policy(model, data, above_box_policy, duration=2.0, framerate=60)

We now have a well-functioning PID controller in configuration space using velocity actuation! It successfully brings us close to the target configuration without overshooting or getting stuck in a steady state before reaching the target. Though the difference between PD and PID controllers may seem subtle in this case, the integral term can be crucial in scenarios where persistent steady-state errors occur due to factors like friction or external disturbances. The integral component helps to correct these errors over time, ensuring that the system reaches and maintains the desired target configuration more accurately.

**Next:** continue to Task 3 in `task3.ipynb`.